In [5]:
!pip install --user jupyter-dash dash dash-leaflet plotly pandas numpy matplotlib pymongo

In [6]:
# Configure the necessary Python module imports for dashboard components
import dash
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### COMPLETE #####
# Change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from AAC import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# COMPLETE Now uses the credentials.json file instead of coded sensitive data

db = AnimalShelter(config_path='credentials.json')

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = dash.Dash(__name__)

#COMPLETE: Add in Grazioso Salvare’s logo
image_filename = 'GraziosoSalvareLogo.png' # replaced with the Grazioso Salvare Logo
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#COMPLETE Place the HTML image tag in the line below into the app.layout code according to your design
#COMPLETE Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('Christian Claudio - CS-340 Dashboard'))),
    html.Center(html.A(html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode())), href='http://www.snhu.edu')),
    html.Hr(),
    html.Div([
        # Added in code for the interactive filtering options using Radio Buttons
        dcc.RadioItems(
            id = 'filter-type',
            options = [
                {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain_wilderness_rescue'},
                {'label': 'Disaster or Individual Tracking', 'value': 'disaster_individual_tracking'},
                {'label': 'Water Rescue', 'value': 'water_rescue'},
                {'label': 'Reset', 'value': 'reset'}
            ],
            value = 'reset',
            labelStyle = {'display': 'inline-block'}
        )
    ]),
    html.Br(),
    # Advanced search filters
    html.Div([
        html.Label(html.B("Search Category Fields: ")),
        html.Div([
            dcc.Input(id='search-breed', type='text', placeholder='Search by Breed', style={'margin-right': '10px', 'width': '250px'}),
            dcc.Input(id='search-name', type='text', placeholder='Search by Animal Name', style={'width': '250px'})
        ], style={'display': 'flex', 'margin-top': '5px'})
    ]),
    html.Hr(),
    dash_table.DataTable(id = 'datatable-id',
                         columns = [{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data = df.to_dict('records'),
    #COMPLETE Set up the features for your interactive data table to make it user-friendly for your client
    #If you completed the Module Six Assignment, you can copy in the code you created here 
                        editable = False,
                        filter_action = "native",
                        sort_action = "native",
                        sort_mode = "multi",
                        column_selectable = "single",
                        row_selectable = "single",
                        row_deletable = False,
                        selected_columns = [],
                        selected_rows = [0],
                        page_action = "native",
                        page_current = 0,
                        page_size = 10,
                        style_table={'overflowX': 'auto'},
                        style_cell={
                            'height': 'auto',
                            'minWidth': '100px', 'width': '100px', 'maxWidth': '200px',
                            'whiteSpace': 'normal'
                        }
                        ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(
    Output('datatable-id','data'),
    [Input('filter-type', 'value'),
     Input('search-breed', 'value'),
     Input('search-name', 'value')]
)
def update_dashboard(filter_type, search_breed, search_name):
    #COMPLETE Add code to filter interactive data table with MongoDB queries
    
    query = {}

    # Check the text searches first
    if search_breed or search_name:
        if search_breed:
            query["breed"] = {"$regex": search_breed, "$options": "i"}
        if search_name:
            query["name"] = {"$regex": search_name, "$options": "i"}
    else:
        if filter_type == 'mountain_wilderness_rescue':
            query = {"animal_type": "Dog", "breed": {"$in": ["German Shepherd", "Alaskan Malamute", 
                                                             "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
                     "sex_upon_outcome": "Intact Male",
                     "age_upon_outcome_in_weeks": {"$gte": 26, "$lte":156}
                    }
        elif filter_type == 'disaster_individual_tracking':
            query = {"animal_type": "Dog", "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever",
                                                      "Bloodhound", "Rottweiler"]},
                     "sex_upon_outcome": "Intact Male",
                     "age_upon_outcome_in_weeks": {"$gte": 20, "$lte":300}
                    }
        elif filter_type == 'water_rescue':
            query = {"animal_type": "Dog", "breed": {"$in": ["Labrador Retriever", "Chesapeake Bay Retriever",
                                                             "Newfoundland"]},
                     "sex_upon_outcome": "Intact Female",
                     "age_upon_outcome_in_weeks": {"$gte": 26, "$lte":156}
                    }
    
    data = pd.DataFrame.from_records(db.read(query))

    if data.empty:
        return []
    
    if '_id' in data.columns:
        data.drop(columns = ['_id'], inplace = True)
    
    return data.to_dict('records')
    
# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    ###FIX ME ####
    # add code for chart of your choice (e.g. pie chart) #
    if viewData is None or len(viewData) == 0:
        return html.Div("There is no data to display")

    data = pd.DataFrame.from_dict(viewData)
    return [
        dcc.Graph(            
            figure = px.pie(data, names='breed', title='Preferred Animals by Breed')
        )    
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):  
    if viewData is None or index is None or len(index) == 0:
        return []
    
    dff = pd.DataFrame.from_dict(viewData)
    row = index[0]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]



app.run(debug=True)

Exception: Failed to load credential: load() missing 1 required positional argument: 'fp'